# Distributed Shared Storage on FABRIC

This notebook demonstrates how to use the **`storage`** flag to automatically mount a distributed shared filesystem on your FABRIC nodes. With a single flag, FABlib handles everything:

1. Adds a FABNetv4 network to each node (required for storage connectivity)
2. Generates your storage credentials via the Storage Service API
3. Uploads credentials and mount scripts to each node
4. Installs required packages and mounts shared storage at `/mnt/cephfs/<cluster>/<user>/`

This notebook showcases three real-world use cases:

| Demo | What It Shows |
|------|---------------|
| **1. Multi-Site Data Sharing** | Write data at one site, read it instantly at another |
| **2. Persistent Storage** | Data survives slice deletion and is available in new slices |
| **3. Data Processing Pipeline** | One node generates data, another processes it via shared storage |

> **New to distributed storage?** Start with the [Quick Start notebook](./cephfs_storage_basic.ipynb) for a minimal example.

### Prerequisites

- Your project must have storage allocated. To request storage, go to **Portal → Experiments → Projects → *Your Project* → Request Storage**.
- Complete the [Configure Environment](../../../configure_and_validate.ipynb) notebook first.

### FABlib API References

- [`fablib.new_slice(storage=True)`](https://fabric-fablib.readthedocs.io/en/latest/fablib.html) — enable storage for all nodes
- [`slice.add_node(storage=True)`](https://fabric-fablib.readthedocs.io/en/latest/slice.html) — enable storage per node
- [`node.enable_storage()`](https://fabric-fablib.readthedocs.io/en/latest/node.html) — enable storage programmatically

## Setup

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.show_config();

### Discover available storage clusters

In [ ]:
import json

clusters = fablib.discover_ceph_clusters()
print(f"Available storage clusters: {json.dumps(clusters, indent=2)}")

user_clusters = fablib.discover_user_ceph_clusters()
print(f"\nClusters with your credentials: {json.dumps(user_clusters, indent=2)}")

---
## Demo 1: Multi-Site Data Sharing

This demo creates two nodes at **different FABRIC sites** (randomly selected). Both nodes mount the same distributed shared filesystem. Data written at one site is instantly visible at the other — no manual file transfer needed.

This is the simplest usage: pass `storage=True` to `new_slice()` and every node gets shared storage automatically.

In [ ]:
# Pick two random sites for multi-site demo
[site1, site2] = fablib.get_random_sites(count=2)
print(f"Using sites: {site1}, {site2}")

# Create a slice with storage enabled on ALL nodes
slice1 = fablib.new_slice(name="CephFS-MultiSite", storage=True)

node_a = slice1.add_node(name="site-a-node", site=site1, cores=4, ram=8, disk=50)
node_b = slice1.add_node(name="site-b-node", site=site2, cores=4, ram=8, disk=50)

slice1.submit();

### Verify storage mounts on both nodes

In [ ]:
slice1 = fablib.get_slice(name="CephFS-MultiSite")

for node in slice1.get_nodes():
    print(f"--- {node.get_name()} (storage={node.has_storage()}, cluster={node.get_storage_cluster()}) ---")
    stdout, stderr = node.execute("df -h | grep ceph")

### Write at Site A, read at Site B

Data written to shared storage at one site is instantly available at the other. This is the key benefit of distributed storage — no `scp`, no `rsync`, no coordination needed.

In [ ]:
node_a = slice1.get_node("site-a-node")
node_b = slice1.get_node("site-b-node")

# Discover the shared storage mount path
stdout, _ = node_a.execute("mount | grep ceph | awk '{print $3}' | head -1", quiet=True)
mount_path = stdout.strip()
print(f"Shared storage mount path: {mount_path}")

# Write a dataset at Site A
print(f"\n--- Writing at {node_a.get_site()} ---")
node_a.execute(f"mkdir -p {mount_path}/demo1")
node_a.execute(f"echo 'Experiment results from {node_a.get_site()}' > {mount_path}/demo1/results.txt")
node_a.execute(f"date >> {mount_path}/demo1/results.txt")
stdout, _ = node_a.execute(f"cat {mount_path}/demo1/results.txt")

# Read the same data at Site B — no copy needed!
print(f"--- Reading at {node_b.get_site()} (same data, no transfer!) ---")
stdout, _ = node_b.execute(f"cat {mount_path}/demo1/results.txt")

# Write back from Site B
print(f"--- {node_b.get_site()} appends to the shared file ---")
node_b.execute(f"echo 'Analysis completed at {node_b.get_site()}' >> {mount_path}/demo1/results.txt")

# Verify at Site A
print(f"--- {node_a.get_site()} sees the update ---")
stdout, _ = node_a.execute(f"cat {mount_path}/demo1/results.txt")

### Measure cross-site throughput

Write a larger file at Site A and measure how quickly Site B can read it through shared storage.

In [ ]:
# Write a 256MB file at Site A
print(f"--- {node_a.get_site()}: Writing 256MB test file ---")
node_a.execute(f"dd if=/dev/urandom of={mount_path}/demo1/testfile_256m bs=1M count=256 status=progress")

# Read it at Site B and measure throughput
print(f"\n--- {node_b.get_site()}: Reading 256MB file from CephFS ---")
node_b.execute(f"dd if={mount_path}/demo1/testfile_256m of=/dev/null bs=1M status=progress")

# Clean up the test file
node_a.execute(f"rm -f {mount_path}/demo1/testfile_256m", quiet=True)

### Clean up Demo 1

In [ ]:
# Clean up demo1 directory but keep slice for Demo 2
node_a.execute(f"rm -rf {mount_path}/demo1", quiet=True)
slice1.delete()

---
## Demo 2: Persistent Storage Across Slices

Distributed storage persists independently of your FABRIC slices. This means you can:
1. Create a slice, write data to shared storage
2. **Delete the slice entirely**
3. Create a brand new slice
4. **Your data is still there!**

This is critical for long-running experiments where slices expire or need to be recreated.

### Step 1: Create first slice and write data

In [ ]:
import datetime

[site1] = fablib.get_random_sites(count=1)
print(f"Writer site: {site1}")

# Create first slice with storage
slice2a = fablib.new_slice(name="CephFS-Persist-A", storage=True)
writer = slice2a.add_node(name="writer", site=site1, cores=4, ram=8, disk=50)
slice2a.submit();

In [ ]:
slice2a = fablib.get_slice(name="CephFS-Persist-A")
writer = slice2a.get_node("writer")

# Find the mount path
stdout, _ = writer.execute("mount | grep ceph | awk '{print $3}' | head -1", quiet=True)
mount_path = stdout.strip()
print(f"CephFS mount: {mount_path}")

# Write important "experiment results" to CephFS
timestamp = datetime.datetime.now().isoformat()
writer.execute(f"mkdir -p {mount_path}/demo2_persistent")
writer.execute(f"echo 'Experiment ID: EXP-2024-042' > {mount_path}/demo2_persistent/experiment.txt")
writer.execute(f"echo 'Created: {timestamp}' >> {mount_path}/demo2_persistent/experiment.txt")
writer.execute(f"echo 'Result: pi ≈ 3.14159265358979' >> {mount_path}/demo2_persistent/experiment.txt")
writer.execute(f"dd if=/dev/urandom of={mount_path}/demo2_persistent/dataset.bin bs=1M count=10 status=none", quiet=True)

print("\nData written to CephFS:")
writer.execute(f"ls -lh {mount_path}/demo2_persistent/")
writer.execute(f"cat {mount_path}/demo2_persistent/experiment.txt")

### Step 2: Delete the slice

The slice is gone. The VM is destroyed. But the data on distributed storage lives on.

In [ ]:
slice2a.delete()
print("Slice 'CephFS-Persist-A' deleted!")

### Step 3: Create a completely new slice and recover the data

A brand new slice at a different site — yet our data is still there on distributed storage.

In [ ]:
# Create a completely new slice — even at a different site!
[site2] = fablib.get_random_sites(count=1, avoid=[site1])
print(f"Reader site: {site2} (different from writer site {site1})")

slice2b = fablib.new_slice(name="CephFS-Persist-B", storage=True)
reader = slice2b.add_node(name="reader", site=site2, cores=4, ram=8, disk=50)
slice2b.submit();

In [ ]:
slice2b = fablib.get_slice(name="CephFS-Persist-B")
reader = slice2b.get_node("reader")

# Find the mount path
stdout, _ = reader.execute("mount | grep ceph | awk '{print $3}' | head -1", quiet=True)
mount_path = stdout.strip()

# The data from the deleted slice is still here!
print("Data recovered from CephFS (written by the DELETED slice):")
reader.execute(f"ls -lh {mount_path}/demo2_persistent/")
reader.execute(f"cat {mount_path}/demo2_persistent/experiment.txt")
stdout, _ = reader.execute(f"md5sum {mount_path}/demo2_persistent/dataset.bin")
print("\nThe 10MB dataset binary is intact!")

### Clean up Demo 2

In [ ]:
reader.execute(f"rm -rf {mount_path}/demo2_persistent", quiet=True)
slice2b.delete()

---
## Demo 3: Data Processing Pipeline

This demo shows a real-world pattern: a **producer** node generates a dataset and a **consumer** node processes it — all through shared storage with no file transfers.

Here we use `add_node(storage=True)` to selectively enable storage only on the nodes that need it (node-level control).

In [ ]:
# Pick two random sites for the pipeline
[site1, site2] = fablib.get_random_sites(count=2)
print(f"Producer: {site1}, Consumer: {site2}")

# Node-level storage: only producer and consumer get CephFS
slice3 = fablib.new_slice(name="CephFS-Pipeline")

# Producer generates data (needs CephFS)
producer = slice3.add_node(name="producer", site=site1, cores=4, ram=8, disk=50, storage=True)

# Consumer processes data (needs CephFS)
consumer = slice3.add_node(name="consumer", site=site2, cores=4, ram=8, disk=50, storage=True)

slice3.submit();

### Verify selective storage

In [ ]:
slice3 = fablib.get_slice(name="CephFS-Pipeline")

for node in slice3.get_nodes():
    print(f"--- {node.get_name()} (storage={node.has_storage()}) ---")
    stdout, stderr = node.execute("df -h | grep ceph || echo 'No CephFS mount'")

### Producer: Generate a synthetic dataset

The producer node creates a CSV dataset on shared storage. In a real scenario, this could be sensor data, simulation output, or downloaded datasets.

In [ ]:
producer = slice3.get_node("producer")

# Find the mount path
stdout, _ = producer.execute("mount | grep ceph | awk '{print $3}' | head -1", quiet=True)
mount_path = stdout.strip()

# Generate a synthetic dataset on CephFS
producer.execute(f"mkdir -p {mount_path}/pipeline")

producer.execute(f"""python3 -c "
import csv, random, math
with open('{mount_path}/pipeline/sensor_data.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['timestamp', 'sensor_id', 'temperature', 'humidity', 'pressure'])
    for i in range(10000):
        w.writerow([
            i * 0.1,
            f'sensor_{{random.randint(1,5):02d}}',
            round(20 + 10 * math.sin(i/100) + random.gauss(0, 0.5), 2),
            round(50 + 20 * math.cos(i/150) + random.gauss(0, 1), 2),
            round(1013 + 5 * math.sin(i/200) + random.gauss(0, 0.3), 2),
        ])
print('Dataset generated: 10,000 rows')
" """)

print("Producer output on CephFS:")
producer.execute(f"ls -lh {mount_path}/pipeline/")
producer.execute(f"head -5 {mount_path}/pipeline/sensor_data.csv")

### Consumer: Process the dataset

The consumer reads the producer's CSV directly from shared storage, computes per-sensor statistics, and writes the summary back — all without any file transfer.

In [ ]:
consumer = slice3.get_node("consumer")

consumer.execute(f"""python3 -c "
import csv
from collections import defaultdict

# Read the producer's dataset directly from CephFS
stats = defaultdict(lambda: {{'count': 0, 'temp_sum': 0, 'temp_min': 999, 'temp_max': -999}})
with open('{mount_path}/pipeline/sensor_data.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sid = row['sensor_id']
        temp = float(row['temperature'])
        stats[sid]['count'] += 1
        stats[sid]['temp_sum'] += temp
        stats[sid]['temp_min'] = min(stats[sid]['temp_min'], temp)
        stats[sid]['temp_max'] = max(stats[sid]['temp_max'], temp)

# Write summary back to CephFS
with open('{mount_path}/pipeline/summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['sensor_id', 'count', 'avg_temp', 'min_temp', 'max_temp'])
    for sid in sorted(stats):
        s = stats[sid]
        w.writerow([sid, s['count'], round(s['temp_sum']/s['count'], 2), s['temp_min'], s['temp_max']])

print('Summary written to CephFS!')
" """)

print("Consumer analysis results:")
consumer.execute(f"cat {mount_path}/pipeline/summary.csv")

### Producer sees the consumer's output

The producer can see the summary that the consumer wrote — demonstrating bidirectional data flow through shared storage.

In [ ]:
print("Producer can see the consumer's analysis:")
producer.execute(f"cat {mount_path}/pipeline/summary.csv")

print("\nFull pipeline directory on CephFS:")
producer.execute(f"ls -lh {mount_path}/pipeline/")

### Clean up Demo 3

In [ ]:
producer.execute(f"rm -rf {mount_path}/pipeline", quiet=True)
slice3.delete()

---
## Summary

| Feature | How to Use |
|---------|------------|
| **All nodes get storage** | `fablib.new_slice(name="...", storage=True)` |
| **Selective per-node** | `slice.add_node(name="...", storage=True)` |
| **Specific cluster** | `fablib.new_slice(storage=True, storage_cluster="east")` |
| **Check if enabled** | `node.has_storage()` → `True` / `False` |
| **Get cluster name** | `node.get_storage_cluster()` → `"east"` |

### What happens under the hood

When `storage=True` is set, FABlib automatically:
1. Adds a **FABNetv4** network interface to the node (named `CEPH_STORAGE`)
2. At post-boot time:
   - Discovers available storage clusters (or uses the one you specified)
   - Generates your credentials via the Storage Service API
   - Uploads configuration, keyring, and mount script to the node
   - Installs required packages (handles Rocky 8/9 and Ubuntu)
   - Mounts shared storage at `/mnt/cephfs/<cluster>/<user>/<subvolume>`

### For advanced usage

See the [Distributed Storage Benchmarking notebook](../../complex_recipes/cephfs_benchmarking/) for:
- Manual cluster discovery and credential management
- Write throughput benchmarking (Python, rsync, dd)
- Host network tuning for optimal storage performance
- Shared vs local disk performance comparison